In [1]:
'''
SALGININ ŞİDDETİ (REPRODUCTİON & HASTANE)
reproduction_rate	R değeri — bir hastanın ortalama kaç kişiyi enfekte ettiği
(1'in üstü = yayılıyor, altı = yavaşlıyor)
icu_patients / icu_patients_per_million	
hosp_patients / hosp_patients_per_million	Hastanede yatan hasta sayısı
weekly_icu_admissions / weekly_hosp_admissions	Haftalık yeni yoğun bakım/hastane girişleri
(çok az ülke bu veriyi paylaşmış — dolu satır sayısı çok düşük: ~11.000-24.000)
'''

"\nSALGININ ŞİDDETİ (REPRODUCTİON & HASTANE)\nreproduction_rate\tR değeri — bir hastanın ortalama kaç kişiyi enfekte ettiği\n(1'in üstü = yayılıyor, altı = yavaşlıyor)\nicu_patients / icu_patients_per_million\t\nhosp_patients / hosp_patients_per_million\tHastanede yatan hasta sayısı\nweekly_icu_admissions / weekly_hosp_admissions\tHaftalık yeni yoğun bakım/hastane girişleri\n(çok az ülke bu veriyi paylaşmış — dolu satır sayısı çok düşük: ~11.000-24.000)\n"

In [ ]:
'''
stringency_index	Hükümetin kısıtlama sertliği endeksi (0-100, kapanma/maske zorunluluğu vb.)
hospital_beds_per_thousand	Bin kişiye düşen hastane yatağı
population	Toplam nüfus
'''

'\nstringency_index\tHükümetin kısıtlama sertliği endeksi (0-100, kapanma/maske zorunluluğu vb.)\npopulation_density\tNüfus yoğunluğu (km² başına kişi)\nmedian_age\tMedyan yaş\naged_65_older / aged_70_older\tNüfusun 65/70 yaş üstü yüzdesi\ngdp_per_capita\tKişi başı GSYH\nextreme_poverty\tAşırı yoksulluk oranı\ncardiovasc_death_rate\tKardiyovasküler hastalık ölüm oranı\ndiabetes_prevalence\tDiyabet yaygınlığı\nfemale_smokers / male_smokers\tSigara içme oranları (cinsiyete göre)\nhandwashing_facilities\tEl yıkama imkanına erişim oranı\nhospital_beds_per_thousand\tBin kişiye düşen hastane yatağı\nlife_expectancy\tOrtalama yaşam beklentisi\nhuman_development_index\tİnsani gelişmişlik endeksi (HDI)\npopulation\tToplam nüfus\n'

In [2]:
#gerekli kütüphaneleri yüklüyoruz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#temizlenmiş verileri yükle
df_countries = pd.read_csv(
    r'C:\Users\nazlican\OneDrive\Masaüstü\CoronaVirusProject\data\processed\countries_clean.csv',
    parse_dates=['date']
)

print("Satır-sütun sayısı:", df_countries.shape)
print("Ülke sayısı:", df_countries['location'].nunique())
print("İlk tarih:", df_countries['date'].min())
print("Son tarih:", df_countries['date'].max())

print(
    "Kalan ülke-tarih tekrarı:",
    df_countries.duplicated(#tekrar eden satırları temizliyoruz
        subset=['location', 'date']
    ).sum()
)

Satır-sütun sayısı: (324819, 67)
Ülke sayısı: 194
İlk tarih: 2020-01-01 00:00:00
Son tarih: 2024-08-14 00:00:00
Kalan ülke-tarih tekrarı: 0


In [ ]:
#dolu kayıtları inceliyoruz
spread_coverage = df_countries.groupby('location').agg(
    toplam_satir=('date', 'size'),
    dolu_r_kaydi=('reproduction_rate', 'count'),
    dolu_hastane_hasta_kaydi=('hosp_patients', 'count'),
    dolu_yogun_bakim_kaydi=('icu_patients', 'count'),
    dolu_hastane_milyon_kaydi=(
        'hosp_patients_per_million',
        'count'
    ),
    dolu_yogun_bakim_milyon_kaydi=(
        'icu_patients_per_million',
        'count'
    ),
    dolu_haftalik_hastane_yatisi=(
        'weekly_hosp_admissions',
        'count'
    ),
    dolu_haftalik_yogun_bakim_yatisi=(
        'weekly_icu_admissions',
        'count'
    ),
    dolu_onlem_sikilik_kaydi=(
        'stringency_index',
        'count'
    ),
    dolu_yatak_kaydi=(
        'hospital_beds_per_thousand',
        'count'
    )
)

print(
    spread_coverage
    .sort_values('dolu_r_kaydi', ascending=False)#R değeri en fazla olan ülkeleri getiriyoruz
    .head(20)
)

                toplam_satir  dolu_r_kaydi  dolu_hastane_hasta_kaydi  \
location                                                               
China                   1674          1076                         0   
South Korea             1674          1047                       134   
Japan                   1674          1046                       204   
Italy                   1677          1044                      1627   
Iran                    1674          1041                         0   
Singapore               1674          1038                         0   
France                  1674          1038                      1109   
Germany                 1674          1037                         0   
United Kingdom          1674          1036                      1198   
Spain                   1674          1036                      1045   
United States           1674          1034                      1381   
Switzerland             1674          1033                      

In [6]:
indicator_country_coverage = pd.Series({
    'R değeri bulunan ülke': (
        spread_coverage['dolu_r_kaydi']
        .gt(0)
        .sum()
    ),
    'Hastane hasta verisi bulunan ülke': (
        spread_coverage['dolu_hastane_hasta_kaydi']
        .gt(0)
        .sum()
    ),
    'Yoğun bakım hasta verisi bulunan ülke': (
        spread_coverage['dolu_yogun_bakim_kaydi']
        .gt(0)
        .sum()
    ),
    'Hastane ve yoğun bakım verisi birlikte bulunan ülke': (
        (
            spread_coverage['dolu_hastane_hasta_kaydi'].gt(0)
            & spread_coverage['dolu_yogun_bakim_kaydi'].gt(0)
        )
        .sum()
    ),
    'Haftalık hastane yatışı bulunan ülke': (
        spread_coverage['dolu_haftalik_hastane_yatisi']
        .gt(0)
        .sum()
    ),
    'Haftalık yoğun bakım yatışı bulunan ülke': (
        spread_coverage['dolu_haftalik_yogun_bakim_yatisi']
        .gt(0)
        .sum()
    ),
    'Önlem sıkılığı verisi bulunan ülke': (
        spread_coverage['dolu_onlem_sikilik_kaydi']
        .gt(0)
        .sum()
    ),
    'Yatak kapasitesi bilgisi bulunan ülke': (
        spread_coverage['dolu_yatak_kaydi']
        .gt(0)
        .sum()
    )
})

print(indicator_country_coverage)

R değeri bulunan ülke                                  191
Hastane hasta verisi bulunan ülke                       36
Yoğun bakım hasta verisi bulunan ülke                   38
Hastane ve yoğun bakım verisi birlikte bulunan ülke     32
Haftalık hastane yatışı bulunan ülke                    31
Haftalık yoğun bakım yatışı bulunan ülke                22
Önlem sıkılığı verisi bulunan ülke                     173
Yatak kapasitesi bilgisi bulunan ülke                  172
dtype: int64


In [7]:
#R değerinin incelenmesi
r_valid_data = df_countries.loc[
    df_countries['reproduction_rate'].notna(),
    ['location', 'date', 'reproduction_rate']
].copy()

r_coverage = (
    r_valid_data
    .groupby('location')
    .agg(
        ilk_r_tarihi=('date', 'min'),
        son_r_tarihi=('date', 'max'),
        dolu_r_kaydi=('reproduction_rate', 'count')
    )
)

r_coverage['toplam_satir'] = (
    r_coverage.index
    .map(spread_coverage['toplam_satir'])
)

r_coverage['bos_r_kaydi'] = (
    r_coverage['toplam_satir']
    - r_coverage['dolu_r_kaydi']
)

print(
    r_coverage
    .sort_values('dolu_r_kaydi', ascending=False)
    .head(20)
)

               ilk_r_tarihi son_r_tarihi  dolu_r_kaydi  toplam_satir  \
location                                                               
China            2020-01-23   2023-01-02          1076          1674   
South Korea      2020-02-21   2023-01-02          1047          1674   
Japan            2020-02-22   2023-01-02          1046          1674   
Italy            2020-02-24   2023-01-02          1044          1677   
Iran             2020-02-27   2023-01-02          1041          1674   
France           2020-03-01   2023-01-02          1038          1674   
Singapore        2020-03-01   2023-01-02          1038          1674   
Germany          2020-03-02   2023-01-02          1037          1674   
United Kingdom   2020-03-03   2023-01-02          1036          1674   
Spain            2020-03-03   2023-01-02          1036          1674   
United States    2020-03-05   2023-01-02          1034          1674   
Netherlands      2020-03-06   2023-01-02          1033          

In [8]:
#ilk ve son R kaydı olan ülkelerde arada boş değerler var mı kontrol ediyoruz
r_range_data = (
    df_countries
    .merge(
        r_coverage[
            ['ilk_r_tarihi', 'son_r_tarihi']
        ],
        left_on='location',
        right_index=True,
        how='inner'
    )
)

r_internal_missing = r_range_data.loc[
    (
        r_range_data['date'] >= r_range_data['ilk_r_tarihi']
    )
    & (
        r_range_data['date'] <= r_range_data['son_r_tarihi']
    )
    & r_range_data['reproduction_rate'].isna(),
    ['location', 'date']
]

r_internal_missing_summary = (
    r_internal_missing
    .groupby('location')
    .agg(
        aradaki_bos_r_gunu=('date', 'size'),
        ilk_bos_r_tarihi=('date', 'min'),
        son_bos_r_tarihi=('date', 'max')
    )
    .sort_values('aradaki_bos_r_gunu', ascending=False)
)

print(r_internal_missing_summary.head(20))

Empty DataFrame
Columns: [aradaki_bos_r_gunu, ilk_bos_r_tarihi, son_bos_r_tarihi]
Index: []


In [10]:
#hastane ve yoğun bakım hasta verisinin incelenmesi
hosp_date_coverage = (
    df_countries.loc[
        df_countries['hosp_patients'].notna(),
        [
            'location',
            'date',
            'hosp_patients',
            'hosp_patients_per_million'
        ]
    ]
    .groupby('location')
    .agg(
        ilk_hastane_tarihi=('date', 'min'),
        son_hastane_tarihi=('date', 'max'),
        dolu_hastane_kaydi=('hosp_patients', 'count'),
        dolu_hastane_milyon_kaydi=(
            'hosp_patients_per_million',
            'count'
        )
    )
)

icu_date_coverage = (
    df_countries.loc[
        df_countries['icu_patients'].notna(),
        [
            'location',
            'date',
            'icu_patients',
            'icu_patients_per_million'
        ]
    ]
    .groupby('location')
    .agg(
        ilk_yogun_bakim_tarihi=('date', 'min'),
        son_yogun_bakim_tarihi=('date', 'max'),
        dolu_yogun_bakim_kaydi=('icu_patients', 'count'),
        dolu_yogun_bakim_milyon_kaydi=(
            'icu_patients_per_million',
            'count'
        )
    )
)

hospital_icu_date_coverage = (
    hosp_date_coverage
    .join(icu_date_coverage, how='outer')
)

print(
    hospital_icu_date_coverage
    .sort_values(
        [
            'dolu_hastane_kaydi',
            'dolu_yogun_bakim_kaydi'
        ],
        ascending=False
    )
    .head(20)
)

               ilk_hastane_tarihi son_hastane_tarihi  dolu_hastane_kaydi  \
location                                                                   
Italy                  2020-02-24         2024-08-07              1627.0   
Czechia                2020-03-11         2024-08-12              1616.0   
Malaysia               2020-03-24         2024-08-10              1597.0   
Sweden                 2020-03-13         2024-04-05              1485.0   
Canada                 2020-04-01         2024-03-26              1456.0   
United States          2020-07-15         2024-04-25              1381.0   
Australia              2020-03-31         2023-12-07              1347.0   
Belgium                2020-03-06         2023-06-27              1209.0   
United Kingdom         2020-04-01         2023-07-12              1198.0   
Serbia                 2020-03-07         2023-06-14              1194.0   
Austria                2020-04-01         2023-06-30              1186.0   
Estonia     

In [11]:
#hastane ve yoğun bakım verilerinde kendi aktif tarih aralığında boş günler var mı
def internal_missing_summary(dataframe, metric):
    valid_data = dataframe.loc[
        dataframe[metric].notna(),
        ['location', 'date', metric]
    ]

    metric_coverage = (
        valid_data
        .groupby('location')
        .agg(
            ilk_tarih=('date', 'min'),
            son_tarih=('date', 'max')
        )
    )

    active_range = dataframe.merge(
        metric_coverage,
        left_on='location',
        right_index=True,
        how='inner'
    )

    missing_rows = active_range.loc[
        (
            active_range['date'] >= active_range['ilk_tarih']
        )
        & (
            active_range['date'] <= active_range['son_tarih']
        )
        & active_range[metric].isna(),
        ['location', 'date']
    ]

    return (
        missing_rows
        .groupby('location')
        .agg(
            aradaki_bos_gun=('date', 'size'),
            ilk_bos_tarih=('date', 'min'),
            son_bos_tarih=('date', 'max')
        )
        .sort_values('aradaki_bos_gun', ascending=False)
    )

hosp_internal_missing = internal_missing_summary(
    df_countries,
    'hosp_patients'
)

icu_internal_missing = internal_missing_summary(
    df_countries,
    'icu_patients'
)

print('Hastane verisinde aradaki boşluklar:')
print(hosp_internal_missing.head(20))

print('\nYoğun bakım verisinde aradaki boşluklar:')
print(icu_internal_missing.head(20))

Hastane verisinde aradaki boşluklar:
               aradaki_bos_gun ilk_bos_tarih son_bos_tarih
location                                                  
Japan                     1226    2020-04-29    2024-03-26
Ireland                   1158    2020-03-16    2023-11-25
Bulgaria                  1122    2020-04-27    2023-11-25
Lithuania                  395    2020-10-07    2023-07-08
Cyprus                     321    2022-01-23    2023-04-19
Netherlands                240    2022-04-16    2024-05-26
South Korea                239    2022-03-01    2023-03-06
Portugal                   205    2022-03-14    2022-11-07
Liechtenstein              202    2020-10-16    2022-09-02
Finland                    178    2020-03-22    2021-06-22
Poland                     168    2020-04-09    2022-09-20
Norway                     103    2020-06-14    2022-03-20
Estonia                     47    2020-01-07    2020-02-24
Denmark                     16    2022-08-13    2023-05-21
Malaysia           